# Deep Q-Learning and Actor-Critic for Continuous Lunar Lander

## Overview

This notebook implements a Deep Reinforcement Learning solution for the **Lunar Lander Continuous** environment using **DDPG (Deep Deterministic Policy Gradient)**, which combines Deep Q-Learning concepts with Actor-Critic architecture.

### Problem Statement

The Lunar Lander environment from Box2D requires an agent to land a spacecraft on a flat landing pad.
- **State Space**: 8-dimensional continuous space
- **Action Space**: 2-dimensional continuous space (main engine thrust + side thrusters)
- **Reward**: Positive for landing, negative for crashing

### Why DDPG?

Standard DQN works for discrete action spaces but fails for continuous actions. DDPG combines:
1. **Actor Network**: Learns deterministic policy μ(s)
2. **Critic Network**: Learns Q-value function Q(s,a)
3. **Target Networks**: Stabilize training through soft updates
4. **Replay Buffer**: Breaks temporal correlation
5. **OU Noise**: Continuous exploration

## Step 1: Environment Setup

In [1]:
# Install required packages
!pip install gymnasium gymnasium[box2d] torch numpy matplotlib pygame -q

In [3]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym
from collections import deque
import random
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

env = gym.make('LunarLanderContinuous-v3', render_mode='rgb_array')

state_dim = env.observation_space.shape[0]
action_dim = env.action_space.shape[0]

print(f"State Space Dimension: {state_dim}")
print(f"Action Space Dimension: {action_dim}")

Using device: cuda
State Space Dimension: 8
Action Space Dimension: 2


## Step 2: Understanding the Environment

In [4]:
state, info = env.reset()
print("Initial State:", state)
print("\nState Components:")
print("  [0-1] Position: x, y of the lander")
print("  [2-3] Velocity: x, y velocity")
print("  [4]   Angle: angle of the lander")
print("  [5]   Angular velocity")
print("  [6-7] Contact: left/right leg contact")

random_action = env.action_space.sample()
print("\nRandom Action:", random_action)
print("Action: [0] Main Engine (0-1), [1] Side Engine (-1 to 1)")

Initial State: [ 2.0666122e-04  1.4073906e+00  2.0921197e-02 -1.5686543e-01
 -2.3273140e-04 -4.7389562e-03  0.0000000e+00  0.0000000e+00]

State Components:
  [0-1] Position: x, y of the lander
  [2-3] Velocity: x, y velocity
  [4]   Angle: angle of the lander
  [5]   Angular velocity
  [6-7] Contact: left/right leg contact

Random Action: [0.85271215 0.30025858]
Action: [0] Main Engine (0-1), [1] Side Engine (-1 to 1)


## Step 3: Replay Buffer & Neural Networks

In [5]:
class ReplayBuffer:
    """Experience Replay Buffer - stores transitions for training"""
    def __init__(self, capacity=100000):
        self.buffer = deque(maxlen=capacity)
    
    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))
    
    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            np.array(states, dtype=np.float32),
            np.array(actions, dtype=np.float32),
            np.array(rewards, dtype=np.float32),
            np.array(next_states, dtype=np.float32),
            np.array(dones, dtype=np.float32)
        )
    
    def __len__(self):
        return len(self.buffer)

In [6]:
class Actor(nn.Module):
    """Actor (Policy) Network - maps states to actions"""
    def __init__(self, state_dim, action_dim, hidden_dim=256):
        super(Actor, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim),
            nn.Tanh()
        )
    
    def forward(self, state):
        return self.network(state)


class Critic(nn.Module):
    """Critic (Q-Value) Network - estimates Q(s,a)"""
    def __init__(self, state_dim, action_dim, hidden_dim=256):
        super(Critic, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(state_dim + action_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
    
    def forward(self, state, action):
        return self.network(torch.cat([state, action], dim=1))

## Step 4: OU Noise & DDPG Agent

In [7]:
class OUNoise:
    """Ornstein-Uhlenbeck Noise for continuous exploration"""
    def __init__(self, action_dim, mu=0.0, theta=0.15, sigma=0.2):
        self.action_dim = action_dim
        self.mu = mu
        self.theta = theta
        self.sigma = sigma
        self.state = np.ones(action_dim) * mu
    
    def sample(self):
        self.state += self.theta * (self.mu - self.state) + self.sigma * np.random.randn(self.action_dim)
        return self.state
    
    def reset(self):
        self.state = np.ones(self.action_dim) * self.mu

In [8]:
class DDPGAgent:
    """Deep Deterministic Policy Gradient Agent"""
    
    def __init__(self, state_dim, action_dim, actor_lr=3e-4, critic_lr=3e-4, gamma=0.99, tau=0.005):
        self.gamma = gamma
        self.tau = tau
        
        self.actor = Actor(state_dim, action_dim).to(device)
        self.critic = Critic(state_dim, action_dim).to(device)
        self.actor_target = Actor(state_dim, action_dim).to(device)
        self.critic_target = Critic(state_dim, action_dim).to(device)
        
        self.actor_target.load_state_dict(self.actor.state_dict())
        self.critic_target.load_state_dict(self.critic.state_dict())
        
        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=actor_lr)
        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=critic_lr)
        
        self.replay_buffer = ReplayBuffer()
        self.noise = OUNoise(action_dim)
    
    def select_action(self, state, add_noise=True):
        state = torch.FloatTensor(state).unsqueeze(0).to(device)
        action = self.actor(state).cpu().detach().numpy()[0]
        if add_noise:
            action += self.noise.sample()
        return np.clip(action, -1, 1)
    
    def train_step(self, batch_size=256):
        if len(self.replay_buffer) < batch_size:
            return None
        
        states, actions, rewards, next_states, dones = self.replay_buffer.sample(batch_size)
        
        states = torch.FloatTensor(states).to(device)
        actions = torch.FloatTensor(actions).to(device)
        rewards = torch.FloatTensor(rewards).unsqueeze(1).to(device)
        next_states = torch.FloatTensor(next_states).to(device)
        dones = torch.FloatTensor(dones).unsqueeze(1).to(device)
        
        # Update Critic
        next_actions = self.actor_target(next_states)
        target_q = self.critic_target(next_states, next_actions.detach())
        target_q = rewards + (1 - dones) * self.gamma * target_q
        current_q = self.critic(states, actions)
        critic_loss = nn.MSELoss()(current_q, target_q)
        
        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        self.critic_optimizer.step()
        
        # Update Actor
        actor_loss = -self.critic(states, self.actor(states)).mean()
        
        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        self.actor_optimizer.step()
        
        # Soft update targets
        self.soft_update(self.actor, self.actor_target)
        self.soft_update(self.critic, self.critic_target)
        
        return critic_loss.item(), actor_loss.item()
    
    def soft_update(self, source, target):
        for target_param, param in zip(target.parameters(), source.parameters()):
            target_param.data.copy_(target_param.data * (1.0 - self.tau) + param.data * self.tau)
    
    def save(self, filepath):
        torch.save({'actor': self.actor.state_dict(), 'critic': self.critic.state_dict()}, filepath)
    
    def load(self, filepath):
        checkpoint = torch.load(filepath, map_location=device)
        self.actor.load_state_dict(checkpoint['actor'])
        self.critic.load_state_dict(checkpoint['critic'])

## Step 5: Training Loop

In [9]:
def train(env, agent, num_episodes=1000, max_steps=1000, batch_size=256, reward_threshold=200, warmup_steps=1000):
    """Train the DDPG agent"""
    episode_rewards = []
    episode_losses = []
    
    for episode in range(num_episodes):
        state, _ = env.reset()
        agent.noise.reset()
        episode_reward = 0
        total_loss = 0
        loss_count = 0
        
        for step in range(max_steps):
            if len(agent.replay_buffer) < warmup_steps:
                action = env.action_space.sample()
            else:
                action = agent.select_action(state)
            
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            agent.replay_buffer.push(state, action, reward, next_state, done)
            
            if len(agent.replay_buffer) >= warmup_steps:
                losses = agent.train_step(batch_size)
                if losses:
                    total_loss += losses[0] + losses[1]
                    loss_count += 1
            
            episode_reward += reward
            state = next_state
            
            if done:
                break
        
        avg_loss = total_loss / loss_count if loss_count > 0 else 0
        episode_rewards.append(episode_reward)
        episode_losses.append(avg_loss)
        
        if (episode + 1) % 10 == 0:
            avg_reward = np.mean(episode_rewards[-10:])
            print(f"Episode {episode + 1}/{num_episodes} | Avg Reward: {avg_reward:.2f} | Loss: {avg_loss:.4f}")
        
        if len(episode_rewards) >= 100 and np.mean(episode_rewards[-100:]) >= reward_threshold:
            print(f"\n*** SOLVED! Avg reward: {np.mean(episode_rewards[-100:]):.2f} ***")
            break
    
    return episode_rewards, episode_losses

In [10]:
# Create agent
agent = DDPGAgent(state_dim, action_dim)
print(f"Actor parameters: {sum(p.numel() for p in agent.actor.parameters()):,}")
print(f"Critic parameters: {sum(p.numel() for p in agent.critic.parameters()):,}")

Actor parameters: 68,610
Critic parameters: 68,865


In [ ]:
# Train the agent
print("Starting training...")
print("=" * 60)

episode_rewards, episode_losses = train(
    env=env,
    agent=agent,
    num_episodes=1000,
    max_steps=1000,
    batch_size=256,
    reward_threshold=200,
    warmup_steps=1000
)

print("=" * 60)
print("Training complete!")

Starting training...
Episode 10/1000 | Avg Reward: -297.85 | Loss: 94.8341
Episode 20/1000 | Avg Reward: -218.11 | Loss: 19.3951
Episode 30/1000 | Avg Reward: -278.12 | Loss: 23.2098
Episode 40/1000 | Avg Reward: -113.41 | Loss: -5.6580
Episode 50/1000 | Avg Reward: -86.35 | Loss: -20.4547
Episode 60/1000 | Avg Reward: -61.71 | Loss: -34.5178
Episode 70/1000 | Avg Reward: -96.00 | Loss: -40.9850
Episode 80/1000 | Avg Reward: 24.86 | Loss: -42.0819
Episode 90/1000 | Avg Reward: -5.51 | Loss: -45.9051
Episode 100/1000 | Avg Reward: 29.63 | Loss: -45.2779
Episode 110/1000 | Avg Reward: 78.54 | Loss: -64.0007
Episode 120/1000 | Avg Reward: 15.42 | Loss: -78.9868
Episode 130/1000 | Avg Reward: 8.42 | Loss: -82.5532
Episode 140/1000 | Avg Reward: -9.97 | Loss: -96.0160
Episode 150/1000 | Avg Reward: 29.84 | Loss: -108.1272
Episode 160/1000 | Avg Reward: 64.45 | Loss: -116.7696
Episode 170/1000 | Avg Reward: 41.20 | Loss: -121.2651
Episode 180/1000 | Avg Reward: 81.59 | Loss: -114.5425
Episod

## Step 6: Visualization

In [ ]:
def plot_training(episode_rewards, episode_losses, window=10):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    axes[0, 0].plot(episode_rewards, alpha=0.5)
    axes[0, 0].set_xlabel('Episode')
    axes[0, 0].set_ylabel('Reward')
    axes[0, 0].set_title('Training Rewards (Raw)')
    axes[0, 0].grid(True)
    
    if len(episode_rewards) >= window:
        moving_avg = np.convolve(episode_rewards, np.ones(window)/window, mode='valid')
        axes[0, 1].plot(moving_avg, label=f'Moving Avg (window={window})', color='orange')
    axes[0, 1].axhline(y=200, color='r', linestyle='--', label='Target (200)')
    axes[0, 1].set_xlabel('Episode')
    axes[0, 1].set_ylabel('Reward')
    axes[0, 1].set_title('Training Rewards (Moving Average)')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    axes[1, 0].plot(episode_losses)
    axes[1, 0].set_xlabel('Episode')
    axes[1, 0].set_ylabel('Loss')
    axes[1, 0].set_title('Training Loss')
    axes[1, 0].grid(True)
    
    if len(episode_rewards) >= 100:
        axes[1, 1].hist(episode_rewards[-100:], bins=20, edgecolor='black')
        axes[1, 1].axvline(x=np.mean(episode_rewards[-100:]), color='r', linestyle='--')
        axes[1, 1].set_xlabel('Reward')
        axes[1, 1].set_ylabel('Frequency')
        axes[1, 1].set_title('Reward Distribution (Last 100 Episodes)')
    
    plt.tight_layout()
    plt.savefig('training_results.png', dpi=150)
    plt.show()

plot_training(episode_rewards, episode_losses)

## Step 7: Evaluation

In [ ]:
def evaluate(env, agent, num_episodes=10):
    """Evaluate the agent without exploration noise"""
    eval_rewards = []
    
    for episode in range(num_episodes):
        state, _ = env.reset()
        episode_reward = 0
        done = False
        
        while not done:
            action = agent.select_action(state, add_noise=False)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            episode_reward += reward
            state = next_state
        
        eval_rewards.append(episode_reward)
        print(f"Eval Episode {episode + 1}: Reward = {episode_reward:.2f}")
    
    return np.mean(eval_rewards), np.std(eval_rewards)

print("Evaluating trained agent...")
mean_reward, std_reward = evaluate(env, agent, num_episodes=10)
print(f"\nEvaluation: Mean = {mean_reward:.2f} ± {std_reward:.2f}")

## Step 8: Save & Load Model

In [ ]:
# Save the model
agent.save('ddpg_lunar_lander.pth')
print("Model saved to 'ddpg_lunar_lander.pth'")

# Load and verify
new_agent = DDPGAgent(state_dim, action_dim)
new_agent.load('ddpg_lunar_lander.pth')
test_reward, _ = evaluate(env, new_agent, num_episodes=5)
print(f"Loaded model test reward: {test_reward:.2f}")

## Summary

### DDPG combines Deep Q-Learning + Actor-Critic:

**Deep Q-Learning Components:**
- Experience Replay Buffer
- Target Networks (soft updates)
- Bellman equation for Q-value updates

**Actor-Critic Components:**
- Actor: Learns policy μ(s)
- Critic: Learns Q(s,a)
- Policy gradient updates

**Key Hyperparameters:**
- Actor/Critic LR: 3e-4
- Gamma: 0.99
- Tau: 0.005
- Batch Size: 256
- Replay Buffer: 100,000